In [56]:
import pandas as pd
import numpy as np

In [57]:
densidad = 7850 #kg/m3

Catalogo = pd.read_excel('Perfiles.xlsx', sheet_name='BD')
Familias = Catalogo['Familia'].unique()
print(Familias)

<StringArray>
['I', 'RHS', 'SHS', 'CHS', 'L', 'C', 'Lam']
Length: 7, dtype: str


In [58]:
# --- Funciones de cálculo por familia ---

def calcular_I(df):
    alma = df["h (mm)"] - 2 * df["tf (mm)"]
    df = df.copy()
    df["Area (mm2)"] = alma * df["tw (mm)"] + 2 * df["b (mm)"] * df["tf (mm)"]
    df["Perímetro (mm)"] = 2 * df["h (mm)"] + 4 * df["b (mm)"] - 2 * df["tw (mm)"]
    return df


def calcular_Tub(df):
    df = df.copy()
    df["Area (mm2)"] = (
        2 * df["h (mm)"] * df["tw (mm)"]
        + 2 * df["b (mm)"] * df["tw (mm)"]
        - 4 * df["tw (mm)"]**2
    )
    df["Perímetro (mm)"] = 2 * (df["h (mm)"] + df["b (mm)"])
    return df


def calcular_C(df):
    df = df.copy()
    df["Area (mm2)"] = (
        df["h (mm)"] * df["tw (mm)"]
        + 2 * df["b (mm)"] * df["tf (mm)"]
        - 2 * df["tw (mm)"] * df["tf (mm)"]
    )
    df["Perímetro (mm)"] = 2 * (df["h (mm)"] + 2 * df["b (mm)"])
    return df


def calcular_L(df):
    df = df.copy()
    df["Area (mm2)"] = (
        df["h (mm)"] * df["tw (mm)"]
        + df["b (mm)"] * df["tw (mm)"]
        - df["tw (mm)"] **2
    )
    df["Perímetro (mm)"] = 2 * (df["h (mm)"] + 2 * df["b (mm)"]) + 2 * df["tw (mm)"]
    
    return df

def calcular_Circ(df):
    df = df.copy()
    df["Area (mm2)"] = np.pi * (df["h (mm)"]**2 - (df["h (mm)"] - df["tw (mm)"])**2) / 4
    df["Perímetro (mm)"] = np.pi * df["h (mm)"]
    return df

def calcular_lam(df):
    df = df.copy()
    df["Area (mm2)"] = df['tw (mm)'] * 1000 
    df["Perímetro (mm)"] = 2*df['h (mm)'] + 2*df['tw (mm)'] 
    return df


# --- Mapa familia → función ---
funciones = {
    "I": calcular_I,
    "RHS": calcular_Tub,
    "SHS": calcular_Tub,
    "C": calcular_C,
    "L": calcular_L,
    "CHS": calcular_Circ,
    "Lam": calcular_lam,
}

# --- Aplicar por familia ---
partes = []
for familia, grupo in Catalogo.groupby("Familia"):
    if familia in funciones:
        partes.append(funciones[familia](grupo))
    else:
        print(f"Familia '{familia}' no reconocida — se omite")
        partes.append(grupo)

Catalogo = pd.concat(partes).sort_index()

Catalogo["Peso Calculado (kg/m)"] = Catalogo["Area (mm2)"] * densidad / 1e6
Catalogo["Dif. Peso (%)"] = (Catalogo["W (kg/m)"] - Catalogo["Peso Calculado (kg/m)"]) / Catalogo["W (kg/m)"] * 100
Catalogo["Dif. Peso (%)"] = Catalogo["Dif. Peso (%)"].round(2)
Catalogo["Area Pintura (m2/m)"] = Catalogo["Perímetro (mm)"] / 1000
Catalogo["Area Pintura (m2/ton)"] = Catalogo["Area Pintura (m2/m)"] / Catalogo["W (kg/m)"] * 1000

In [59]:
Catalogo.to_excel('Perfiles_Post.xlsx', sheet_name='BD', index=False)